In [1]:
import sys;sys.path.append('../..')
from abslithist import *

In [3]:
df=pd.read_csv(os.path.expanduser('~/lltk_data/corpora/eebo_tcp/metadata.csv'))
# df=df[df.title.str.lower().str.contains('essay')]
df['decade'] = df.year//10*10
df.decade.value_counts()

decade
1640    9366
1680    8983
1690    7602
1650    6028
1660    5158
        ... 
650        1
660        1
1960       1
5280       1
1860       1
Name: count, Length: 108, dtype: int64

In [4]:
norm_d = get_norm_dict()
# norm_d

In [5]:
def score_freqs(path_json):
    with open(path_json) as f:
        freqs = json.load(f)
    out = []
    for word,count in freqs.items():
        if word in norm_d:
            for n in range(count):
                out.append(norm_d[word])
    return np.mean(out)

In [6]:
folder = os.path.expanduser('~/lltk_data/corpora/eebo_tcp/freqs/')
folder

'/Users/ryan/lltk_data/corpora/eebo_tcp/freqs/'

In [7]:
def score_freqs_folder(folder, total=None):
    out = []
    iterr = tqdm(total=total)
    for root,dirs,files in os.walk(folder):
        for file in files:
            if file.endswith('.json'):
                iterr.update(1)
                path_json = os.path.join(root,file)
                score = score_freqs(path_json)
                out.append({'id':path_json.replace(folder,'').replace('.json',''), 'score':score})
    return pd.DataFrame(out)

In [8]:
df_scores = score_freqs_folder(folder, total=len(df))
df_scores

100%|██████████| 56351/56351 [01:13<00:00, 767.71it/s]


,id,score
0,A90571,-0.752249
1,A26145,-0.642927
2,A59448,-0.343362
3,B00188,0.070423
4,A21680,-0.358487
...,...,...
56346,A40538,-0.074006
56347,A89401,-0.208295
56348,A42505,-0.655193
56349,A69655,-0.710170


In [9]:
odf=df_scores.dropna().merge(df, on='id', how='left').set_index('id').sort_values('score',ascending=False)
odf

,score,_llp_,author,corpus,date,extent,fnfn_xml,genre,id_eebo_citation,id_oclc,...,id_stc,id_vid,medium,num_words,ocr_accuracy,publisher,pubplace,title,year,decade
id,,,,,,,,,,,,,,,,,,,,,
A62551,1.396194,eebo_tcp|A62551,"Tillinghast, Mary.",EEBO-TCP,"printed in the year, 1690.","[2], 30 p.",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,99833001.0,NaN,...,ESTC R221736,37476.0,Prose,4008,0.975549,"[s.n.],",London :,"Rare and excellent receipts Experienc'd, and t...",1690,1690
A80288,1.338004,eebo_tcp|A80288,NaN,EEBO-TCP,1694.,"[4], 450, [22] p. :",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,99896382.0,NaN,...,ESTC R229840,154131,Prose,52330,0.977451,"printed, and sold by G. Conyers at the Golden ...",London :,"The compleat cook: or, the whole art of cooker...",1694,1690
A52399,1.334410,eebo_tcp|A52399,"Norfolk, Henry Howard, Duke of, 1655-1701.",EEBO-TCP,1685.,1 sheet ([1] p.),/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,11862243.0,ocm 11862243,...,ESTC R17510,50037.0,Prose,557,0.992819,"Printed by Nat. Thompson ...,",[London] :,The Duke of Norfolk's order about the habit th...,1685,1680
A53974,1.325299,eebo_tcp|A53974,NaN,EEBO-TCP,1674.,"[6], 450, [22] p.",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,99830841.0,NaN,...,ESTC R219363,35295.0,Prose,67728,0.980540,"printed for Simon Miller at the Star, at the w...",London :,The English and French cook describing the bes...,1674,1670
A80290,1.307093,eebo_tcp|A80290,NaN,EEBO-TCP,1690.,"[4], 450, [22] p.",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,99897192.0,NaN,...,ESTC R224403,170821,Prose,73027,0.982390,"printed for William Miller, at the Gilded Acor...",London :,The compleat English and French cook describin...,1690,1690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A47321,-1.165036,eebo_tcp|A47321,"Keynes, John, 1625?-1697.",EEBO-TCP,Printed in the year 1674.,"[30], 124 p.",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,99825062.0,NaN,...,ESTC R200380,29434.0,Prose,15624,0.994816,"s.n.],",[London :,"A rational, compendious way to convince, witho...",1674,1670
A36261,-1.168439,eebo_tcp|A36261,"Dodwell, Henry, 1641-1711.",EEBO-TCP,1676.,"[88], 31, 21, 120 [1] p.",/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,11781896.0,ocm 11781896,...,ESTC R1351,49109.0,Prose,49029,0.993534,"Printed for Benj. Tooke ...,",London :,Two short discourses against the Romanists by ...,1676,1670
A58497,-1.182083,eebo_tcp|A58497,NaN,EEBO-TCP,1648?],1 sheet ([1] p.),/Volumes/Present/DH/corpora/eebo/_xml_eebo_tcp...,Prose,12827448.0,ocm 12827448,...,ESTC N12688,94292.0,Prose,430,0.986047,"s.n.,",[London :,"Remarks on the Quakers case, deliver'd to the ...",1648,1640


In [10]:
odf.to_pickle('../../data/scores/v3/data.scores.EEBO_TCP.pkl')